In [0]:
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql import DataFrame

In [0]:
def read_raw_data(
    storage_account: str,
    folder_name : str,
    file_name: str = "*",
    file_format: str = "csv",
    adf_run_id: str = "default_run_id",
    extra_read_options : dict | None = None
) -> DataFrame:

    raw_path = f"/Volumes/alwaha_banking_dev_001/bronze/raw_files_volume/{folder_name}"
    schema_path = f"/Volumes/alwaha_banking_dev_001/bronze/raw_files_volume/_schemas/{folder_name}"

    default_csv_option = {
        
        "header": "true",
        "cloudFiles.schemaLocation": schema_path,
        "pathGlobFilter": file_name

        }
    default_json_option = {
        "multiline": "true",
        "cloudFiles.schemaLocation": schema_path,
        "pathGlobFilter": file_name

        }
    finnal_options = {}

    if file_format.lower() == "csv":
        finnal_options.update(default_csv_option)

    elif file_format.lower() == "json":
        finnal_options.update(default_json_option)
    
    if extra_read_options:
        finnal_options.update(extra_read_options)

    
    df_raw = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", file_format)
              .options(**finnal_options)
              .load(raw_path)
              )

    df_finnal = df_raw.withColumns({
        "_ingested_at": current_timestamp(),
        "source_file": col("_metadata.file_path"),
        "adf_run_id" : lit(adf_run_id) 
    })

    return df_finnal